# RAG pipeline

In [1]:
import os
import sys
from dotenv import load_dotenv
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
    set_global_handler
)
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_index.core.prompts import PromptTemplate

# logging
import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("llama_index.core").setLevel(logging.WARNING)
logging.getLogger("fsspec").setLevel(logging.WARNING)
set_global_handler("simple")

load_dotenv()

c:\Users\chamu\code\user-manual-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Setup LLM and Embedding model

In [ ]:
llm = GoogleGenAI(model="gemini-2.5-flash-lite")
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = llm
Settings.embed_model = embed_model

documents = SimpleDirectoryReader("../data").load_data()

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3750.37it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INFO:sentence_transformers.SentenceTransformer:1 prompt is loaded, with the key: query


## Chunk the document and store the RAG index

NOTE: If the chunking strategy is changed it will NOT automatically update your existing stored index. It loads the old index with old chunks
* To apply a new chunking strategy, delete the old storage folder.

In [ ]:
# chunking
splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print("Chunking documents into nodes...")
nodes = splitter.get_nodes_from_documents(documents)

# vector index
if os.path.exists("../storage"):
    print("Loading index from storage...")
    storage_context = StorageContext.from_defaults(persist_dir="../storage")
    index = load_index_from_storage(storage_context)
else:
    print("Creating new index...")
    index = VectorStoreIndex.from_documents(nodes)
    index.storage_context.persist("../storage")

Chunking documents into nodes...
Loading index from storage...


In [4]:
# basic retriever
retriever = index.as_retriever(similarity_top_k=5)

print("Querying the index...")
query_engine = RetrieverQueryEngine.from_args(
    retriever,
    llm=llm
)

print("Generating response...")
response = await query_engine.aquery(
    "What is the name of this device?"
)
print(response)

Querying the index...
Generating response...
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
** Messages: **
user: Context information is below.
---------------------
file_path: C:\Users\chamu\code\user-manual-rag\data\User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf

<</D(User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.78 0 R/Title<FEFF0053006F0066007400770061007200650020004F007000650072006100740069006F006E>>>0069006F006E00200061006E0064002000410073007300690067006E006D0065006E00740073>>>

file_path: C:\Users\chamu\code\user-manual-rag\data\User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf

uʖZ+e,>&      [E
<</BBox[0.0 841.89 595.276 0.0]/Filter/FlateDecode/Length 240/Matrix[1.0 0.0 0.0 1.0 0.0 0.0]/Resources<</ExtGState<</GS0 3327 0 R/GS1 3326 0 R>>/Font<</TT0 241 0 R>>/ProcSet[/PDF/Text]>>/Subtype/Form>>stream
HK1+Qk3*`sْvf֋R3o`x	Yc1$F͓4oF-;nL9ip]<60J9g,}Mlmďv3UTa|eg}YُdyN*FD:IhMV0vzro(E)	;HV溭x[b|Nѻdv.

file_path: C:\Users\chamu\code\user-manual-ra

# RAG Evaluation

In [5]:
eval_dataset = [
    {
        "question": "What is the name of this device?",
        "ground_truth": "XSM2 SE Camera"
    },
    {
        "question": "How many indicators are there on the side of the device that show the device status?",
        "ground_truth": "Three"
    }
]

In [6]:
def evaluate_retrieval(retriever, eval_dataset):
    results = []

    for sample in eval_dataset:
        nodes = retriever.retrieve(sample["question"])
        contexts = [n.text for n in nodes]

        hit = any(sample["ground_truth"].lower() in c.lower() for c in contexts)

        results.append({
            "question": sample["question"],
            "hit": hit,
            "contexts": contexts
        })

    hit_rate = sum(r["hit"] for r in results) / len(results)
    print(f"Retrieval Hit Rate: {hit_rate:.2f}")

    return results

In [7]:
judge_prompt = PromptTemplate(
    """You are evaluating an answer.

Question: {question}
Ground Truth: {ground_truth}
Answer: {answer}

Is the answer acceptable? Respond with ONLY 'yes' or 'no'.
"""
)

async def evaluate_with_llm(query_engine, eval_dataset, llm):
    scores = []

    for sample in eval_dataset:
        response = await query_engine.aquery(sample["question"])
        answer = str(response)

        judge_input = judge_prompt.format(
            question=sample["question"],
            ground_truth=sample["ground_truth"],
            answer=answer
        )

        judgment = await llm.acomplete(judge_input)
        is_correct = "yes" in judgment.text.lower()

        scores.append(is_correct)

    accuracy = sum(scores) / len(scores)
    print(f"LLM Judged Accuracy: {accuracy:.2f}")

In [8]:
evaluate_retrieval(retriever, eval_dataset)
await evaluate_with_llm(query_engine, eval_dataset, llm)

Retrieval Hit Rate: 0.00
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
** Messages: **
user: Context information is below.
---------------------
file_path: C:\Users\chamu\code\user-manual-rag\data\User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf

<</D(User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.78 0 R/Title<FEFF0053006F0066007400770061007200650020004F007000650072006100740069006F006E>>>0069006F006E00200061006E0064002000410073007300690067006E006D0065006E00740073>>>

file_path: C:\Users\chamu\code\user-manual-rag\data\User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf

uʖZ+e,>&      [E
<</BBox[0.0 841.89 595.276 0.0]/Filter/FlateDecode/Length 240/Matrix[1.0 0.0 0.0 1.0 0.0 0.0]/Resources<</ExtGState<</GS0 3327 0 R/GS1 3326 0 R>>/Font<</TT0 241 0 R>>/ProcSet[/PDF/Text]>>/Subtype/Form>>stream
HK1+Qk3*`sْvf֋R3o`x	Yc1$F͓4oF-;nL9ip]<60J9g,}Mlmďv3UTa|eg}YُdyN*FD:IhMV0vzro(E)	;HV溭x[b|Nѻdv.

file_path: C:\Users\chamu\code\user-manual-rag\data\User_Manual_S